<a href="https://colab.research.google.com/github/SAMYSOSERIOUS/Master-Thesis/blob/main/research/SOTA/mrkr_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1 · Setup

In [ ]:
import subprocess, sys, importlib
subprocess.run([sys.executable,'-m','pip','install','-q','cleanlab'], check=False)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, json, gc, re, math, random
import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from PIL import Image
PROJECT = Path('/content/drive/MyDrive/Master Thesis')
sys.path.insert(0, str(PROJECT/'scope3'))
import config; importlib.reload(config)
import training_lib_max as TM
device = 'cuda' if torch.cuda.is_available() else 'cpu'; assert device=='cuda','Switch to GPU runtime.'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print('TM loaded:', hasattr(TM,'prepare_local_data'), '| GPU:', torch.cuda.get_device_name(0))

Mounted at /content/drive
TM loaded: True | GPU: Tesla T4


## 2 · Config

In [ ]:
OUTDIR = PROJECT/'mrkr_benchmark'; CKPT = OUTDIR/'ckpts'
for d in [OUTDIR,CKPT]: d.mkdir(parents=True, exist_ok=True)
BACKBONE   = 'base'           # base suits 40k images; same as OAI winning model
RES        = 224              # keep 224 — corrected labels defined at 224px
CLEAN_MODE = 'aggressive'     # same protocol as OAI 82.1%
CAP_CW     = 4.0              # MRKR is more balanced than NHANES; mild cap
SPLIT_SEED = 0
TEST_FRAC, VAL_FRAC_OF_REST = 0.15, 0.1765
EPOCHS, PATIENCE, MIN_EPOCHS, LR = 45, 8, 8, 3e-4
print(f'BACKBONE={BACKBONE} · CLEAN_MODE={CLEAN_MODE} · RES={RES}')

BACKBONE=base · CLEAN_MODE=aggressive · RES=224


## 3 · Training core

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix,
                              balanced_accuracy_score, recall_score)
from sklearn.model_selection import train_test_split, GroupKFold
from cleanlab.filter import find_label_issues

class ArrayDS(Dataset):
    def __init__(self, df, images, train=False, size=RES):
        self.df=df.reset_index(drop=True); self.images=images; self.train=train
        self.aug=(T.Compose([T.RandomHorizontalFlip(0.5), T.RandomRotation(10),
                             T.RandomResizedCrop(size,scale=(0.85,1.0),antialias=True),
                             T.RandomApply([T.ColorJitter(0.2,0.2)],p=0.3)]) if train
                  else T.Compose([T.Resize((size,size),antialias=True)]))
        self.norm=T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; arr=np.asarray(self.images[int(r['arr_idx'])]).astype(np.float32)/255.0
        x=torch.from_numpy(arr).unsqueeze(0).repeat(3,1,1)
        return self.norm(self.aug(x)), int(r['kl_grade'])

def build_convnext(arch='base', device='cuda'):
    if arch=='small': m=models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
    elif arch=='tiny': m=models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    else: m=models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
    m.classifier[2]=nn.Linear(m.classifier[2].in_features,5); return m.to(device)

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay=decay; self.shadow={k:v.detach().clone() for k,v in model.state_dict().items()}
    def update(self, model):
        for k,v in model.state_dict().items():
            if v.dtype.is_floating_point: self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
            else: self.shadow[k]=v.detach().clone()
    def state(self): return {k:v.clone() for k,v in self.shadow.items()}

@torch.no_grad()
def predict(model, ds_maker, df, device='cuda', bs=64, tta=False):
    model.eval(); out=[]
    ld=DataLoader(ds_maker(df,train=False), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    for x,_ in ld:
        x=x.to(device,non_blocking=True)
        with torch.amp.autocast('cuda'):
            p=model(x).softmax(1)
            if tta: p=p+model(torch.flip(x,[3])).softmax(1)
        out.append((p/(2 if tta else 1)).float().cpu().numpy())
    return np.concatenate(out)

def train_generic(train_df,val_df,*,build_fn,ds_maker,class_weights=None,epochs=45,patience=8,
                  min_epochs=1,bs=32,lr=3e-4,warmup=3,accum=2,seed=0,device='cuda',log=print):
    torch.manual_seed(seed)
    tr=DataLoader(ds_maker(train_df,train=True),batch_size=bs,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
    model=build_fn(device=device)
    head=[p for n,p in model.named_parameters() if 'classifier' in n]
    body=[p for n,p in model.named_parameters() if 'classifier' not in n]
    opt=torch.optim.AdamW([{'params':body,'lr':lr*0.1},{'params':head,'lr':lr}],weight_decay=1e-4)
    from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
    sched=SequentialLR(opt,[LinearLR(opt,0.1,1.0,total_iters=warmup),
                            CosineAnnealingLR(opt,T_max=max(1,epochs-warmup))],milestones=[warmup])
    cw=torch.tensor(class_weights,dtype=torch.float32,device=device) if class_weights is not None else None
    crit=nn.CrossEntropyLoss(weight=cw,label_smoothing=0.05); scaler=torch.amp.GradScaler('cuda'); ema=EMA(model,0.999)
    best=-1; best_state=None; wait=0; yv=val_df['kl_grade'].values
    for ep in range(1,epochs+1):
        model.train(); opt.zero_grad(set_to_none=True)
        for bi,(x,y) in enumerate(tr):
            x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
            with torch.amp.autocast('cuda'): loss=crit(model(x),y)/accum
            scaler.scale(loss).backward()
            if (bi+1)%accum==0: scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); ema.update(model)
        sched.step()
        tmp=build_fn(device=device); tmp.load_state_dict(ema.state()); pv=predict(tmp,ds_maker,val_df,device); del tmp
        acc=accuracy_score(yv,pv.argmax(1)); qwk=cohen_kappa_score(yv,pv.argmax(1),weights='quadratic')
        log(f'  ep{ep:02d}/{epochs} val_acc={acc:.4f} qwk={qwk:.3f}'+('  *best*' if acc>best+1e-4 else f'  ({wait+1}/{patience})'))
        if acc>best+1e-4: best=acc; best_state=ema.state(); wait=0
        else:
            wait+=1
            if wait>=patience and ep>=min_epochs: log(f'  early stop ep{ep}, best={best:.4f}'); break
    del model,tr; gc.collect(); torch.cuda.empty_cache()
    return best_state,best

def robust_fold(train_df,val_df,*,min_quality=0.40,**kw):
    log=kw.get('log',print)
    st,best=train_generic(train_df,val_df,seed=kw.pop('seed',0),min_epochs=6,**kw)
    if best<min_quality:
        log(f'  WEAK FOLD ({best:.3f}) — retry...'); st,best=train_generic(train_df,val_df,seed=999,min_epochs=6,**kw)
        log(f'  retry best={best:.3f}')
    return st,best
print('Core ready.')

Core ready.


## 4 · Stage 1 — load MRKR from packed array

In [ ]:
manifest=TM.prepare_local_data(); images=TM._IMAGES
print('datasets:', dict(manifest['dataset'].value_counts()))
cands=[d for d in manifest['dataset'].unique() if any(k in str(d).lower() for k in ['mrkr','mrk','mark'])]
assert cands, f'MRKR key not found. Keys: {list(manifest["dataset"].unique())}'
KEY=cands[0]
df=manifest[manifest['dataset']==KEY].copy().reset_index(drop=True)
df['orig_kl']=df['kl_grade'].astype(int)
if 'patient_id' in df.columns: df['grp']=df['patient_id'].astype(str)
else:
    ext=df['filename'].astype(str).str.extract(r'(\d{5,})')[0]
    df['grp']=ext.fillna(pd.Series(df.index.astype(str)))
print(f'MRKR key: {KEY} | rows: {len(df)} | groups: {df["grp"].nunique()}')
print('KL distribution:', dict(df['orig_kl'].value_counts().sort_index()))
print('KL %:', {k:f'{v*100:.1f}%' for k,v in df['orig_kl'].value_counts(normalize=True).sort_index().items()})
mkMR=lambda d,train: ArrayDS(d,images,train=train,size=RES)

# guard: ensure cleaning folds will be viable
n=len(df); print(f'\ncleaning viability: {n} images / 3 folds = {n//3} per fold')
assert n//3 >= 5000, f'Too few images per fold ({n//3}); cleaning quality may be poor'
print('OK — large enough for reliable cleaning (cf. NHANES: 1595/fold → unreliable)')

Copied array in 111s
Loaded array (61558, 224, 224) in 12s
datasets: {'mrkr': np.int64(39967), 'oai': np.int64(8547), 'mendeley': np.int64(8259), 'nhanes3': np.int64(4785)}
MRKR key: mrkr | rows: 39967 | groups: 31311
KL distribution: {0: np.int64(13639), 1: np.int64(2675), 2: np.int64(13893), 3: np.int64(6041), 4: np.int64(3719)}
KL %: {0: '34.1%', 1: '6.7%', 2: '34.8%', 3: '15.1%', 4: '9.3%'}

cleaning viability: 39967 images / 3 folds = 13322 per fold
OK — large enough for reliable cleaning (cf. NHANES: 1595/fold → unreliable)


## 5 · Stage 2 — fixed patient-level split (saved)

In [ ]:
SPLIT_JSON=OUTDIR/'mrkr_split.json'
if SPLIT_JSON.exists():
    sp=json.loads(SPLIT_JSON.read_text()); TRAIN_G,VAL_G,TEST_G=set(sp['train']),set(sp['val']),set(sp['test'])
    print(f'loaded split -> train {len(TRAIN_G)} | val {len(VAL_G)} | test {len(TEST_G)}')
else:
    g=df.groupby('grp')['orig_kl'].max().reset_index().rename(columns={'orig_kl':'pmax'}).sort_values('grp').reset_index(drop=True)
    def safe_split(frame,frac,seed):
        strat=frame['pmax'] if (frame['pmax'].value_counts()>=2).all() else None
        return train_test_split(frame,test_size=frac,random_state=seed,stratify=strat)
    trv,te=safe_split(g,TEST_FRAC,SPLIT_SEED); tr,va=safe_split(trv,VAL_FRAC_OF_REST,SPLIT_SEED)
    TRAIN_G,VAL_G,TEST_G=set(tr['grp']),set(va['grp']),set(te['grp'])
    SPLIT_JSON.write_text(json.dumps({'train':sorted(TRAIN_G),'val':sorted(VAL_G),'test':sorted(TEST_G)}))
    print(f'created + saved -> train {len(TRAIN_G)} | val {len(VAL_G)} | test {len(TEST_G)}')
for a,b in [(TRAIN_G,VAL_G),(TRAIN_G,TEST_G),(VAL_G,TEST_G)]: assert a.isdisjoint(b),'LEAK!'
df['split']=np.where(df.grp.isin(TEST_G),'test',np.where(df.grp.isin(VAL_G),'val','train'))
print('images per split:', dict(df['split'].value_counts()))

created + saved -> train 21916 | val 4698 | test 4697
images per split: {'train': np.int64(28034), 'test': np.int64(5978), 'val': np.int64(5955)}


## 6 · Stage 3 — CleanLab OOF (cached)

Cleaning folds run on the FULL dataset (same as OAI). Each fold needs to reach val_acc ≥ 0.55 to be trusted.
**If any fold stays below 0.45, it will auto-retry.** Watch the fold quality — it's the signal that cleaning is reliable.

In [ ]:
labels=df['orig_kl'].values.astype(int)
OOS=OUTDIR/'mrkr_oos.npy'
if OOS.exists() and np.load(OOS,mmap_mode='r').shape[0]==len(df):
    oos=np.load(OOS); print(f'loaded cached OOS {oos.shape}')
else:
    freq=df['orig_kl'].value_counts().sort_index()
    cwc=np.minimum(len(df)/(5.0*np.array([freq.get(k,1) for k in range(5)])),CAP_CW); cwc=(cwc/cwc.mean()).tolist()
    oos=np.zeros((len(df),5),dtype=np.float32); fold_quality=[]
    for fi,(tri,vai) in enumerate(GroupKFold(3).split(df,groups=df['grp'].values)):
        print(f'\nclean fold {fi+1}/3  ({len(tri)} train | {len(vai)} val)')
        st,best=robust_fold(df.iloc[tri],df.iloc[vai],build_fn=lambda device:build_convnext(BACKBONE,device),
                            ds_maker=mkMR,class_weights=cwc,epochs=18,patience=4,bs=48,accum=1,lr=LR,device=device,log=print)
        fold_quality.append(best)
        if best<0.55: print(f'  WARNING: fold {fi+1} val_acc={best:.3f} — cleaning quality may be degraded')
        m=build_convnext(BACKBONE,device); m.load_state_dict(st)
        oos[vai]=predict(m,mkMR,df.iloc[vai],device); del m; gc.collect(); torch.cuda.empty_cache()
    print(f'\nfold quality: {[round(q,3) for q in fold_quality]}')
    assert min(fold_quality)>=0.45, f'Cleaning folds too weak (min={min(fold_quality):.3f}) — do not trust relabeling'
    np.save(OOS,oos); print(f'saved OOS -> {OOS}')

flagged=find_label_issues(labels=labels,pred_probs=oos,return_indices_ranked_by='self_confidence')
alt=oos.argmax(1); altc=oos.max(1); corr=labels.copy()
for i in flagged: corr[i]=alt[i]
df['clean_kl']=corr; n_ch=int((corr!=labels).sum())
print(f'flagged {len(flagged)} | relabeled {n_ch} ({100*n_ch/len(df):.1f}%)')
print('NOTE: MRKR labels are model-generated (Duke). Corrected accuracy = agreement with relabeled model labels, NOT clinical accuracy.')


clean fold 1/3  (26644 train | 13323 val)
Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:02<00:00, 122MB/s]


  ep01/18 val_acc=0.1976 qwk=0.149  *best*
  ep02/18 val_acc=0.3369 qwk=0.535  *best*
  ep03/18 val_acc=0.4741 qwk=0.635  *best*
  ep04/18 val_acc=0.5347 qwk=0.675  *best*
  ep05/18 val_acc=0.5494 qwk=0.690  *best*
  ep06/18 val_acc=0.5597 qwk=0.702  *best*
  ep07/18 val_acc=0.5704 qwk=0.706  *best*
  ep08/18 val_acc=0.5740 qwk=0.709  *best*
  ep09/18 val_acc=0.5719 qwk=0.712  (1/4)
  ep10/18 val_acc=0.5742 qwk=0.717  *best*
  ep11/18 val_acc=0.5728 qwk=0.719  (1/4)
  ep12/18 val_acc=0.5742 qwk=0.717  (2/4)
  ep13/18 val_acc=0.5751 qwk=0.716  *best*
  ep14/18 val_acc=0.5729 qwk=0.715  (1/4)
  ep15/18 val_acc=0.5730 qwk=0.713  (2/4)
  ep16/18 val_acc=0.5736 qwk=0.710  (3/4)
  ep17/18 val_acc=0.5740 qwk=0.709  (4/4)
  early stop ep17, best=0.5751

clean fold 2/3  (26645 train | 13322 val)
  ep01/18 val_acc=0.2003 qwk=0.176  *best*
  ep02/18 val_acc=0.3205 qwk=0.527  *best*
  ep03/18 val_acc=0.4732 qwk=0.642  *best*
  ep04/18 val_acc=0.5110 qwk=0.672  *best*
  ep05/18 val_acc=0.5351 qwk=0

## 7 · Stage 4 — train ConvNeXt-Base @224 + TTA (saved)

In [ ]:
tr_df=df[df.split=='train'].copy(); va_df=df[df.split=='val'].copy(); te_df=df[df.split=='test'].copy()
tr_df['kl_grade']=tr_df['clean_kl'].astype(int); va_df['kl_grade']=va_df['clean_kl'].astype(int)
fr=pd.Series(tr_df['kl_grade']).value_counts().sort_index()
cw=np.minimum(len(tr_df)/(5.0*np.array([fr.get(k,1) for k in range(5)])),CAP_CW); cw=(cw/cw.mean()).tolist()
print(f'train {len(tr_df)} | val {len(va_df)} | test {len(te_df)} | class weights {[round(x,2) for x in cw]}')

ck=CKPT/f'convnext_{BACKBONE}_{CLEAN_MODE}.pt'
if ck.exists():
    print('checkpoint found -> loading'); st=torch.load(ck,map_location=device)
else:
    st,best=train_generic(tr_df,va_df,build_fn=lambda device:build_convnext(BACKBONE,device),ds_maker=mkMR,
                          class_weights=cw,epochs=EPOCHS,patience=PATIENCE,min_epochs=MIN_EPOCHS,
                          bs=32,accum=2,lr=LR,device=device,log=print)
    torch.save(st,ck); print(f'saved -> {ck}  (best val_acc {best:.4f})')
model=build_convnext(BACKBONE,device); model.load_state_dict(st); model.eval()
probs=predict(model,mkMR,te_df,device,tta=True); pred=probs.argmax(1); del model; gc.collect()

train 28034 | val 5955 | test 5978 | class weights [0.58, 1.04, 0.64, 1.03, 1.72]
  ep01/45 val_acc=0.2423 qwk=0.118  *best*
  ep02/45 val_acc=0.4131 qwk=0.592  *best*
  ep03/45 val_acc=0.5718 qwk=0.752  *best*
  ep04/45 val_acc=0.6767 qwk=0.826  *best*
  ep05/45 val_acc=0.7523 qwk=0.872  *best*
  ep06/45 val_acc=0.7955 qwk=0.897  *best*
  ep07/45 val_acc=0.8198 qwk=0.912  *best*
  ep08/45 val_acc=0.8326 qwk=0.920  *best*
  ep09/45 val_acc=0.8380 qwk=0.922  *best*
  ep10/45 val_acc=0.8435 qwk=0.923  *best*
  ep11/45 val_acc=0.8475 qwk=0.922  *best*
  ep12/45 val_acc=0.8452 qwk=0.921  (1/8)
  ep13/45 val_acc=0.8460 qwk=0.919  (2/8)
  ep14/45 val_acc=0.8474 qwk=0.921  (3/8)
  ep15/45 val_acc=0.8494 qwk=0.921  *best*
  ep16/45 val_acc=0.8507 qwk=0.924  *best*
  ep17/45 val_acc=0.8521 qwk=0.924  *best*
  ep18/45 val_acc=0.8539 qwk=0.925  *best*
  ep19/45 val_acc=0.8529 qwk=0.923  (1/8)
  ep20/45 val_acc=0.8514 qwk=0.922  (2/8)
  ep21/45 val_acc=0.8519 qwk=0.924  (3/8)
  ep22/45 val_acc=0.8

10

## 8 · Stage 5 — results

In [ ]:
yo=te_df['orig_kl'].values.astype(int); yc=te_df['clean_kl'].values.astype(int)
acc_c=accuracy_score(yc,pred); acc_o=accuracy_score(yo,pred)
bacc=balanced_accuracy_score(yo,pred); qwk_c=cohen_kappa_score(yc,pred,weights='quadratic')
n=len(yc); ci=1.96*math.sqrt(acc_c*(1-acc_c)/n)
rec=recall_score(yo,pred,average=None,labels=[0,1,2,3,4],zero_division=0)

print('='*62); print(f'MRKR · ConvNeXt-{BACKBONE} @ {RES} + TTA · FIRST BENCHMARK'); print('='*62)
print(f'  corrected acc (headline)    : {acc_c*100:.2f}%  ±{ci*100:.1f}   (n={n})')
print(f'  QWK (corrected)             : {qwk_c:.3f}')
print(f'  relabeled                   : {n_ch} ({100*n_ch/len(df):.1f}%)')
print(f'  --- honesty: original model-generated labels ---')
print(f'  original-label accuracy     : {acc_o*100:.2f}%   (agreement with Duke labeler)')
print(f'  balanced accuracy           : {bacc*100:.2f}%')
print(f'  per-class recall            : '+'  '.join(f'KL{k}={rec[k]*100:.0f}%' for k in range(5)))
print('\nconfusion matrix (rows = corrected-true KL 0..4):'); print(confusion_matrix(yc,pred,labels=[0,1,2,3,4]))

json.dump({'dataset':'MRKR','note':'labels are model-generated (Duke), NOT radiologist',
           'protocol':'CleanLab-aggressive + ConvNeXt-base + TTA','backbone':f'convnext_{BACKBONE}',
           'acc_corrected':float(acc_c),'acc_original':float(acc_o),'balanced_acc':float(bacc),
           'qwk_corrected':float(qwk_c),'ci95':float(ci),'n':int(n),'relabel_pct':100*n_ch/len(df)},
          open(OUTDIR/'mrkr_benchmark_summary.json','w'),indent=2)
print(f'\nsaved -> {OUTDIR/"mrkr_benchmark_summary.json"}')

MRKR · ConvNeXt-base @ 224 + TTA · FIRST BENCHMARK
  corrected acc (headline)    : 85.06%  ±0.9   (n=5978)
  QWK (corrected)             : 0.921
  relabeled                   : 13739 (34.4%)
  --- honesty: original model-generated labels ---
  original-label accuracy     : 58.31%   (agreement with Duke labeler)
  balanced accuracy           : 58.96%
  per-class recall            : KL0=65%  KL1=49%  KL2=48%  KL3=63%  KL4=69%

confusion matrix (rows = corrected-true KL 0..4):
[[1545  148   52    1    9]
 [ 106  850   85    6    2]
 [  63  155 1262   91   11]
 [   1    9   52  881   49]
 [   8    7    5   33  547]]

saved -> /content/drive/MyDrive/Master Thesis/mrkr_benchmark/mrkr_benchmark_summary.json
